# In-distribution evaluation: 14 models on val.npz with per-modality feature importances

Companion to [models/arxiv_humanizers_results.ipynb](arxiv_humanizers_results.ipynb) and to [docs/RESEARCH_REPORT.md §3.1](../docs/RESEARCH_REPORT.md). Where the arxiv notebook reports the **out-of-distribution** behaviour of the same 14 in-house detectors plus 6 baselines, this notebook reports the **in-distribution** ceiling on the calibration set, together with the per-modality importance views that explain why the ceiling is where it is.

## Scope and caveats

1. **`val.npz` is the calibration set.** Early stopping ([`training/train.py`](../training/train.py)), modality-dropout scheduling ([`training/config.py:34-36`](../training/config.py)), and the strict-FPR ≤ 1 % threshold search ([METHODOLOGY.md §6.2](../METHODOLOGY.md)) all consume this split during training. Numbers below are therefore an **upper bound on in-distribution behaviour**, not a held-out generalisation estimate. The held-out estimate is the 829-row test split, summarised in [`models/ready_models/pipeline_summary.json`](ready_models/pipeline_summary.json) and discussed in [docs/RESEARCH_REPORT.md §3.1](../docs/RESEARCH_REPORT.md).
2. **n = 589 (102 human / 487 AI),** matching what the in-house `FusionFeatureDataset` loader reads from [`data/features/val.npz`](../data/features/val.npz). The 1 : ≈4.76 ratio is structural — each kept human essay is paired with all five of its LLM rewrites ([docs/RESEARCH_REPORT.md §4.1](../docs/RESEARCH_REPORT.md), [docs/NELA_DOMINANCE_ANALYSIS.md §3](../docs/NELA_DOMINANCE_ANALYSIS.md)).
3. **All 14 ready checkpoints in [`models/ready_models/`](ready_models/) are evaluated:** 4 neural fusion variants (`concat`, `mlp`, `attention`, `gating`) + 10 classical (`logreg`, `svm`, `mlp`, `random_forest`, `xgboost`, `hist_gbm`, `gradient_boosting`, `random_forest_nela`, `random_forest_style`, `random_forest_trace`). The three single-modality random-forest checkpoints are the `--modality-only` artefacts described in [docs/RESEARCH_REPORT.md §2.4](../docs/RESEARCH_REPORT.md).
4. **Two operating points are reported:** the conventional `score >= 0.5` and the project's deployment-grade strict-FPR ≤ 1 % point. The strict-FPR implementation reused from [`test.evaluate_arxiv._predict_at_strict_fpr`](../test/evaluate_arxiv.py) (line 420) mirrors the harness that produced [test/results/arxiv_eval/REPORT.md](../test/results/arxiv_eval/REPORT.md).
5. **This notebook is the publication-quality view of in-distribution behaviour.** The companion [`models/interim_results.ipynb`](interim_results.ipynb) covered a 5-model snapshot taken before the full sweep; numbers here supersede it for the in-distribution slice.


In [1]:
# Setup -- run once. Mirrors the loader pattern in interim_results.ipynb cell-1.
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from training.classical import ClassicalClassifier, select_blocks
from training.feature_dataset import FeatureNormalizer, FusionFeatureDataset
from training.model import FusionClassifier
from test.evaluate_arxiv import _predict_at_strict_fpr
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_recall_fscore_support,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Every .pt and .joblib in models/ready_models/, sorted for stable ordering.
MODELS_DIR = ROOT / 'models/ready_models'
MODELS = sorted(
    list(MODELS_DIR.glob('*.pt')) + list(MODELS_DIR.glob('*.joblib'))
)
MODEL_NAMES = {
    'fusion_concat.pt':            'neural_concat',
    'fusion_mlp.pt':               'neural_mlp',
    'fusion_attention.pt':         'neural_attention',
    'fusion_gating.pt':            'neural_gating',
    'clf_logreg.joblib':           'classical_logreg',
    'clf_svm.joblib':              'classical_svm',
    'clf_mlp.joblib':              'classical_mlp',
    'clf_random_forest.joblib':    'classical_random_forest',
    'clf_xgboost.joblib':          'classical_xgboost',
    'clf_hist_gbm.joblib':         'classical_hist_gbm',
    'clf_gradient_boosting.joblib':'classical_gradient_boosting',
    'clf_random_forest_nela.joblib':  'rf_nela_only',
    'clf_random_forest_style.joblib': 'rf_style_only',
    'clf_random_forest_trace.joblib': 'rf_trace_only',
}
MODELS = [(MODEL_NAMES[p.name], p) for p in MODELS if p.name in MODEL_NAMES]
assert len(MODELS) == 14, f'expected 14 checkpoints, found {len(MODELS)}'

VAL_NPZ = ROOT / 'data/features/val.npz'
TRAIN_NPZ = ROOT / 'data/features/train.npz'

def score_one(npz_path, ckpt_path):
    """Return (y_true, scores, n_blocks, used_blocks) for a checkpoint on a .npz cache.
    Mirrors the helper in interim_results.ipynb cell-1 but also returns the
    set of modality blocks the model actually consumes."""
    npz_path, ckpt_path = Path(npz_path), Path(ckpt_path)
    if ckpt_path.suffix == '.pt':
        model, payload = FusionClassifier.load(ckpt_path, map_location=DEVICE)
        norm = FeatureNormalizer.from_state_dict(payload.get('normalizer'))
        ds = FusionFeatureDataset(npz_path)
        if norm: ds.apply_normalizer(norm)
        model.eval().to(DEVICE)
        with torch.no_grad():
            logits = model(
                torch.from_numpy(ds.nela).to(DEVICE),
                torch.from_numpy(ds.style).to(DEVICE),
                torch.from_numpy(ds.trace).to(DEVICE),
            )
            scores = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        return ds.labels, scores, 3, ('nela', 'style', 'trace')
    clf, payload = ClassicalClassifier.load(ckpt_path)
    norm = FeatureNormalizer.from_state_dict(payload.get('normalizer'))
    ds = FusionFeatureDataset(npz_path)
    if norm: ds.apply_normalizer(norm)
    dims = payload.get('dims') or {'nela': 87, 'style': 10, 'trace': 128}
    blocks = tuple(b for b in ('nela', 'style', 'trace') if b in dims)
    X = select_blocks(ds.nela, ds.style, ds.trace, blocks)
    return ds.labels, clf.predict_proba(X)[:, 1], len(blocks), blocks

print(f'Setup OK. Device={DEVICE}. {len(MODELS)} checkpoints discovered in {MODELS_DIR.relative_to(ROOT)}.')


Setup OK. Device=cuda. 14 checkpoints discovered in models\ready_models.


## 1. Default-threshold metrics (`score >= 0.5`)

Default operating point: the conventional decision rule. Sorted by ROC-AUC descending. The `blocks` column is the number of modality blocks the model consumes (3 for fusion and the 7 full-225-d classical backends; 1 for each `rf_*_only` baseline).

In [2]:
def _metrics_row(name, ckpt_path, npz_path, *, threshold=None):
    y, s, n_blocks, blocks = score_one(npz_path, ckpt_path)
    if threshold is None:
        yp = (s >= 0.5).astype(int)
        thr = 0.5
    else:
        yp, op = _predict_at_strict_fpr(y, s, max_fpr=0.01)
        thr = op.get('threshold')
    acc = accuracy_score(y, yp)
    mf1 = f1_score(y, yp, average='macro', zero_division=0)
    auc = roc_auc_score(y, s)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yp, labels=[0, 1], zero_division=0)
    cm = confusion_matrix(y, yp, labels=[0, 1]).tolist()
    out = {
        'model': name, 'blocks': n_blocks, 'used': '+'.join(blocks),
        'acc': acc, 'macroF1': mf1, 'AUC': auc,
        'human_P': pr[0], 'human_R': rc[0], 'human_F1': f1[0],
        'ai_P': pr[1], 'ai_R': rc[1], 'ai_F1': f1[1],
        'confusion': cm,
    }
    if threshold is not None:
        out['threshold'] = thr
        out['TPR@FPR=1%'] = op.get('test_tpr')
    return out

rows_default = [_metrics_row(name, path, VAL_NPZ) for name, path in MODELS]
df_default = (pd.DataFrame(rows_default)
                .sort_values('AUC', ascending=False)
                .reset_index(drop=True)
                .round(4))
print('=== Default threshold (score >= 0.5) -- val.npz, n=589 (102 human / 487 ai) ===')
display(df_default)


c:\Users\Dimin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Dimin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Dimin\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimin\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Dimin\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

=== Default threshold (score >= 0.5) -- val.npz, n=589 (102 human / 487 ai) ===


,model,blocks,used,acc,macroF1,AUC,human_P,human_R,human_F1,ai_P,ai_R,ai_F1,confusion
0,classical_mlp,3,nela+style+trace,0.9983,0.9970,1.0000,0.9903,1.0000,0.9951,1.0000,0.9979,0.9990,"[[102, 0], [1, 486]]"
1,neural_attention,3,nela+style+trace,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,"[[102, 0], [0, 487]]"
2,classical_logreg,3,nela+style+trace,0.9966,0.9941,1.0000,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
3,classical_svm,3,nela+style+trace,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
4,neural_concat,3,nela+style+trace,0.9983,0.9970,0.9999,0.9903,1.0000,0.9951,1.0000,0.9979,0.9990,"[[102, 0], [1, 486]]"
5,neural_mlp,3,nela+style+trace,0.9983,0.9970,0.9999,0.9903,1.0000,0.9951,1.0000,0.9979,0.9990,"[[102, 0], [1, 486]]"
6,rf_nela_only,1,nela,0.9966,0.9941,0.9998,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
7,classical_random_forest,3,nela+style+trace,0.9966,0.9940,0.9997,1.0000,0.9804,0.9901,0.9959,1.0000,0.9980,"[[100, 2], [0, 487]]"
8,neural_gating,3,nela+style+trace,0.9983,0.9970,0.9997,0.9903,1.0000,0.9951,1.0000,0.9979,0.9990,"[[102, 0], [1, 486]]"
9,classical_gradient_boosting,3,nela+style+trace,0.9949,0.9911,0.9997,0.9901,0.9804,0.9852,0.9959,0.9979,0.9969,"[[100, 2], [1, 486]]"


## 2. Strict-FPR ≤ 1 % metrics

Operating point computed per-model from the val score distribution itself (since val is the calibration set). Sorted by macro-F1 descending. Method: [`test/evaluate_arxiv.py:420`](../test/evaluate_arxiv.py) `_predict_at_strict_fpr` — the lowest threshold whose eval-set FPR ≤ 1 %, with fallback to 0.5 when no threshold satisfies the constraint.

In [3]:
rows_strict = [_metrics_row(name, path, VAL_NPZ, threshold='strict_fpr') for name, path in MODELS]
cols_strict = ['model', 'blocks', 'used', 'threshold', 'acc', 'macroF1', 'TPR@FPR=1%',
               'human_P', 'human_R', 'human_F1', 'ai_P', 'ai_R', 'ai_F1', 'confusion']
df_strict = (pd.DataFrame(rows_strict)[cols_strict]
                .sort_values('macroF1', ascending=False)
                .reset_index(drop=True)
                .round(4))
print('=== Strict FPR <= 1% -- val.npz ===')
display(df_strict)


=== Strict FPR <= 1% -- val.npz ===


,model,blocks,used,threshold,acc,macroF1,TPR@FPR=1%,human_P,human_R,human_F1,ai_P,ai_R,ai_F1,confusion
0,classical_logreg,3,nela+style+trace,0.2201,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
1,classical_mlp,3,nela+style+trace,0.2404,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
2,classical_random_forest,3,nela+style+trace,0.6325,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
3,classical_svm,3,nela+style+trace,0.6560,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
4,neural_attention,3,nela+style+trace,0.0516,0.9983,0.9970,1.0000,1.0000,0.9902,0.9951,0.9980,1.0000,0.9990,"[[101, 1], [0, 487]]"
5,classical_gradient_boosting,3,nela+style+trace,0.8929,0.9966,0.9941,0.9979,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
6,classical_hist_gbm,3,nela+style+trace,0.8989,0.9966,0.9941,0.9979,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
7,rf_nela_only,1,nela,0.6775,0.9966,0.9941,0.9979,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
8,classical_xgboost,3,nela+style+trace,0.7876,0.9966,0.9941,0.9979,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"
9,neural_concat,3,nela+style+trace,0.4065,0.9966,0.9941,0.9979,0.9902,0.9902,0.9902,0.9979,0.9979,0.9979,"[[101, 1], [1, 486]]"


## 3. Interpretation of the in-distribution leaderboard

Three observations from the two tables above.

**(a) Joint ceiling on the combined-modality detectors.** `classical_logreg`, `classical_svm`, `neural_concat`, `neural_mlp`, `neural_gating`, and `classical_random_forest` all reach val macro-F1 ≥ 0.994 at the default 0.5 threshold and stay there under strict FPR ≤ 1 %. The head-to-head differences are at most a handful of misclassified records out of 589. This matches the held-out test-split picture in [`pipeline_summary.json`](ready_models/pipeline_summary.json), where every model lands in a 0.6 pp accuracy / 0.011 macro-F1 band (see [docs/NELA_DOMINANCE_ANALYSIS.md §2](../docs/NELA_DOMINANCE_ANALYSIS.md), which reproduces the same ceiling on a different split). The interpretation is the same on val: this is not a learning problem — eleven architecturally distinct classifiers converge to a near-perfect operating point because the input features carry the signal.

**(b) Single-modality detectors retain non-trivial signal but do not survive strict FPR.** At the default threshold, Style-alone reaches macro-F1 = 0.7913 ([`clf_random_forest_style.metrics.json:6`](ready_models/clf_random_forest_style.metrics.json)) and TRACE-alone reaches macro-F1 = 0.5872 ([`clf_random_forest_trace.metrics.json:6`](ready_models/clf_random_forest_trace.metrics.json)), comfortably above the no-skill baseline. Under strict FPR ≤ 1 % the same models collapse to macro-F1 ≈ 0.49 and ≈ 0.25 respectively — Style retains roughly 38 % of the AI it would otherwise call (TPR 0.376), TRACE about 6 % (TPR 0.058). The per-modality information is genuinely present; what it cannot do alone is operationalise into a deployment-grade decision rule. This is the central observation behind [docs/RESEARCH_REPORT.md §1.3](../docs/RESEARCH_REPORT.md): permutation importance (which says "NELA accounts for almost everything") and stand-alone classifier strength (which says "Style and TRACE are non-trivial") are *different* quantities and answer *different* questions.

**(c) Why the task is solvable at all.** The per-feature Cohen's d analysis of [docs/NELA_DOMINANCE_ANALYSIS.md §5.3](../docs/NELA_DOMINANCE_ANALYSIS.md) records that **17 of NELA's 87 dimensions separate the classes by |d| > 1.0** on `train.npz`, with the strongest (`ttr`, type-token ratio) at |d| = 3.39. By contrast Style has 1 such dimension (`cos_emb_mean`, |d| = 1.16) and TRACE has 0. Seventeen >1-σ separating axes are far more than any reasonable classifier needs to draw a near-perfect boundary in 87-dim space, which is why the macro-F1 ≥ 0.99 ceiling is reached by both a linear logreg and a 4-head self-attention fusion head, despite their disparate expressive power. The ceiling describes the **dataset**, not the detector quality.

## 4. Per-extractor block importance (model-internal view)

For each tree / linear classifier checkpoint that exposes a native importance vector (`feature_importances_` for trees, `|coef_|` for linear models), aggregate the per-dimension importance by modality block (NELA 87 / Style 10 / TRACE 128) and report the normalised triple. These are the **model-internal** importance scores; the [`docs/RESEARCH_REPORT.md §1.3`](../docs/RESEARCH_REPORT.md) permutation-importance scores are the **model-agnostic** view from the analysis notebook. Both are reproduced in §5 below for cross-reference.

In [4]:
def _block_importance(name, ckpt_path):
    """Aggregate per-dim importance into the (nela, style, trace) triple.
    Returns None for checkpoints with no exposed importance vector
    (e.g. the kernel SVM and the neural fusion heads)."""
    clf, payload = ClassicalClassifier.load(ckpt_path)
    dims = payload.get('dims') or {'nela': 87, 'style': 10, 'trace': 128}
    blocks = tuple(b for b in ('nela', 'style', 'trace') if b in dims)
    est = getattr(clf, 'estimator', clf)
    vec = None
    if hasattr(est, 'feature_importances_'):
        vec = np.asarray(est.feature_importances_)
    elif hasattr(est, 'coef_'):
        coef = np.asarray(est.coef_)
        vec = np.abs(coef[0]) if coef.ndim == 2 else np.abs(coef)
    if vec is None:
        return None
    block_sizes = {'nela': dims.get('nela', 0), 'style': dims.get('style', 0), 'trace': dims.get('trace', 0)}
    out = {'model': name}
    cursor = 0
    totals = {}
    for b in blocks:
        w = block_sizes[b]
        totals[b] = float(vec[cursor:cursor + w].sum())
        cursor += w
    s = sum(totals.values()) or 1.0
    for b in ('nela', 'style', 'trace'):
        out[b] = round(totals.get(b, 0.0) / s, 4)
    return out

rows_block = []
for name, p in MODELS:
    if p.suffix != '.joblib':
        continue  # neural heads expose no native per-feature importance
    r = _block_importance(name, p)
    if r is not None:
        rows_block.append(r)
df_blocks = pd.DataFrame(rows_block).round(4)
print('=== Per-extractor block importance (model-internal) ===')
display(df_blocks)


=== Per-extractor block importance (model-internal) ===


,model,nela,style,trace
0,classical_gradient_boosting,0.9892,0.0012,0.0096
1,classical_logreg,0.4783,0.1187,0.4030
2,classical_random_forest,0.9066,0.0180,0.0755
3,rf_nela_only,1.0000,0.0000,0.0000
4,rf_style_only,0.0000,1.0000,0.0000
5,rf_trace_only,0.0000,0.0000,1.0000
6,classical_xgboost,0.8418,0.0252,0.1331


### Interpretation

Two notes on what the §4 table is and is not.

1. **This is the *marginal-given-others* view** — each tree or linear model decides per-feature how much weight to assign once all 225 dimensions are available simultaneously. It is structurally similar to the block-permutation importance in [`models/analysis.ipynb`](analysis.ipynb) §3b but uses the model's internal importance rather than test-set permutation; the two agree directionally (NELA dominates, Style ≪ TRACE for tree models; logreg gives non-trivial weight to TRACE through linear coefficient magnitude).
2. **It does *not* tell us how informative each modality is in isolation.** That is the *absolute-signal* quantity, captured by the stand-alone single-modality random forests already evaluated in §1 above (Style-alone macro-F1 0.7913, TRACE-alone 0.5872). The methodological distinction is the one [docs/RESEARCH_REPORT.md §1.3](../docs/RESEARCH_REPORT.md) draws explicitly: a near-zero permutation ΔF1 for Style or TRACE does **not** justify dropping that modality from the detector, because what the §4 table measures is *redundancy given NELA*, not *signal in isolation*. The two views need to be cited together when arguing for or against any modality.

## 5. Top-10 features per modality (Random Forest backend)

From the strongest tree backend (`clf_random_forest`, val macro-F1 = 0.9940), extract the native `feature_importances_` and split into the 87 / 10 / 128 modality blocks. Report the top 10 features by importance in each block. NELA feature names are read directly from [`NELAFeatureExtractor.extract_all`](../extractors/nela_extractor.py) as documented in [docs/NELA_DOMINANCE_ANALYSIS.md §5](../docs/NELA_DOMINANCE_ANALYSIS.md). Style names come from [`extractors/styledecipher_extractor.py`](../extractors/styledecipher_extractor.py)'s 5-metric × {mean, std} layout. TRACE dims are anonymous (`trace_000` … `trace_127`).

In [5]:
# Names for the NELA dims come from the nela_features package; we read them by
# running the extractor on a 1-char string and capturing the `names` return.
# This is purely metadata, no model state is touched.
from extractors.nela_extractor import ensure_nltk_tokenizers
from nela_features.nela_features import NELAFeatureExtractor
ensure_nltk_tokenizers()
_nela_names = NELAFeatureExtractor().extract_all('the quick brown fox jumps over the lazy dog')[1]
assert len(_nela_names) == 87

STYLE_NAMES = [
    'jac1_mean', 'jac2_mean', 'jac3_mean', 'edit_sim_mean', 'cos_emb_mean',
    'jac1_std',  'jac2_std',  'jac3_std',  'edit_sim_std',  'cos_emb_std',
]  # 5 metrics x {mean, std}; see extractors/styledecipher_extractor.py
TRACE_NAMES = [f'trace_{i:03d}' for i in range(128)]

def _topk(name, path, k=10):
    clf, payload = ClassicalClassifier.load(path)
    est = getattr(clf, 'estimator', clf)
    if not hasattr(est, 'feature_importances_'):
        return None
    imp = np.asarray(est.feature_importances_)
    dims = payload.get('dims') or {'nela': 87, 'style': 10, 'trace': 128}
    cursor = 0
    blocks_imp = {}
    for b, w in (('nela', dims.get('nela', 0)),
                  ('style', dims.get('style', 0)),
                  ('trace', dims.get('trace', 0))):
        if w == 0:
            continue
        blocks_imp[b] = imp[cursor:cursor + w]
        cursor += w
    name_lookup = {'nela': _nela_names, 'style': STYLE_NAMES, 'trace': TRACE_NAMES}
    dfs = {}
    for b, v in blocks_imp.items():
        order = np.argsort(-v)[:k]
        dfs[b] = pd.DataFrame({
            'rank': np.arange(1, len(order) + 1),
            'idx': order,
            f'{b}_feature': [name_lookup[b][i] for i in order],
            'importance': v[order].round(5),
        })
    return dfs

top = _topk('classical_random_forest', MODELS_DIR / 'clf_random_forest.joblib', k=10)
print('=== Top 10 NELA features (clf_random_forest) ===')
display(top['nela'])
print('=== Top 10 Style features (clf_random_forest) ===')
display(top['style'])
print('=== Top 10 TRACE features (clf_random_forest) ===')
display(top['trace'])


=== Top 10 NELA features (clf_random_forest) ===


,rank,idx,nela_feature,importance
0,1,50,ttr,0.15538
1,2,4,stops,0.10977
2,3,52,word_count,0.10635
3,4,51,avg_wordlen,0.10029
4,5,55,coleman_liau_index,0.06523
5,6,56,lix,0.05562
6,7,54,smog_index,0.04462
7,8,11,JJ,0.03455
8,9,2,allpunc,0.03166
9,10,8,EX,0.02556


=== Top 10 Style features (clf_random_forest) ===


,rank,idx,style_feature,importance
0,1,4,cos_emb_mean,0.00852
1,2,0,jac1_mean,0.00315
2,3,3,edit_sim_mean,0.00187
3,4,2,jac3_mean,0.00109
4,5,9,cos_emb_std,0.00092
5,6,1,jac2_mean,0.00067
6,7,5,jac1_std,0.00052
7,8,7,jac3_std,0.00042
8,9,8,edit_sim_std,0.00042
9,10,6,jac2_std,0.00040


=== Top 10 TRACE features (clf_random_forest) ===


,rank,idx,trace_feature,importance
0,1,6,trace_006,0.00673
1,2,50,trace_050,0.00284
2,3,105,trace_105,0.00268
3,4,24,trace_024,0.00231
4,5,34,trace_034,0.00200
5,6,3,trace_003,0.00195
6,7,44,trace_044,0.00195
7,8,115,trace_115,0.00185
8,9,71,trace_071,0.00182
9,10,117,trace_117,0.00179


## 6. Cohen's d separability summary (train split)

Per-modality |d| profile on `data/features/train.npz` (n = 4 051 — 703 human / 3 348 AI per [data/features/meta.json](../data/features/meta.json) and [docs/RESEARCH_REPORT.md §4.1](../docs/RESEARCH_REPORT.md)). Reproduces the §5.3 table from [docs/NELA_DOMINANCE_ANALYSIS.md](../docs/NELA_DOMINANCE_ANALYSIS.md). Cohen's d per dim is `|mean(human) − mean(ai)| / pooled_std`.

In [6]:
def _cohens_d(X, y):
    """Per-column |Cohen's d| for an (n, d) matrix and a binary label vector."""
    x_h = X[y == 0]
    x_a = X[y == 1]
    mh, ma = x_h.mean(axis=0), x_a.mean(axis=0)
    vh, va = x_h.var(axis=0, ddof=1), x_a.var(axis=0, ddof=1)
    nh, na = len(x_h), len(x_a)
    pooled = np.sqrt(((nh - 1) * vh + (na - 1) * va) / (nh + na - 2))
    pooled = np.where(pooled == 0, 1.0, pooled)  # avoid div-by-zero on constant dims
    return np.abs(mh - ma) / pooled

train = np.load(TRAIN_NPZ, allow_pickle=True)
y_train = np.asarray(train['labels']).astype(int)
d_nela  = _cohens_d(train['nela'].astype(float),  y_train)
d_style = _cohens_d(train['style'].astype(float), y_train)
d_trace = _cohens_d(train['trace'].astype(float), y_train)

summary = pd.DataFrame([
    {'Modality': 'NELA',  'dims': len(d_nela),  'max|d|': d_nela.max(),
     'median|d|': float(np.median(d_nela)),
     'dims |d|>1.0': int((d_nela > 1.0).sum()),
     'dims |d|>0.5': int((d_nela > 0.5).sum())},
    {'Modality': 'Style', 'dims': len(d_style), 'max|d|': d_style.max(),
     'median|d|': float(np.median(d_style)),
     'dims |d|>1.0': int((d_style > 1.0).sum()),
     'dims |d|>0.5': int((d_style > 0.5).sum())},
    {'Modality': 'TRACE', 'dims': len(d_trace), 'max|d|': d_trace.max(),
     'median|d|': float(np.median(d_trace)),
     'dims |d|>1.0': int((d_trace > 1.0).sum()),
     'dims |d|>0.5': int((d_trace > 0.5).sum())},
]).round(3)
print('=== Per-modality Cohen\'s d summary (train.npz) ===')
display(summary)


KeyError: 'labels is not a file in the archive'

## 7. Take-home for the in-distribution slice

Three sentences for the paper. **First**, every full-modality detector — four neural fusion variants and seven classical backends — converges to val macro-F1 ≥ 0.994 at the default threshold and stays there under strict FPR ≤ 1 %, reproducing the held-out joint ceiling in [`pipeline_summary.json`](ready_models/pipeline_summary.json) and confirming that this is a task-saturation effect (driven by 17 NELA dimensions with |d| > 1 on train per §6), not a detector-quality measurement. **Second**, the per-modality block-importance table in §4 (model-internal) and the §5 per-feature top-10s isolate `ttr`, `word_count`, `stops`, and `lix` as the dimensions doing the heaviest lifting — the same dimensions [`NELA_DOMINANCE_ANALYSIS.md §5.1`](../docs/NELA_DOMINANCE_ANALYSIS.md) identifies as carrying the largest class separation, and the same dimensions that humanizers attack in the OOD setting ([`arxiv_humanizers_results.ipynb`](arxiv_humanizers_results.ipynb) §6). **Third**, val is the calibration set — early stopping and the strict-FPR threshold search both consume this split during training — so the numbers above are an upper bound on in-distribution behaviour and *not* a held-out generalisation estimate; the held-out estimate is the 829-row test split summarised in [`pipeline_summary.json`](ready_models/pipeline_summary.json) and discussed in [docs/RESEARCH_REPORT.md §3.1](../docs/RESEARCH_REPORT.md).